# 05 — CF-Miner: Úloha 1
## Sezónní profil výkonnosti
### 4IZ503 Projektový seminář — Ultra Marathon Running

---

### Slovní zadání

Existují kombinace sezóny + dalších atributů, kde je rozložení `speed_cat`
výrazně odlišné od průměru populace?

**Hypotéza:** Zimní závody favorizují zkušenější závodníky (nováčci
odpadají z pole), zatímco jarní závody mají vyrovnanější profil.
Letní závody na dlouhých trasách by měly mít odlišný profil
díky extrémním podmínkám.

**Target:** `speed_cat` (3 kategorie: pomalý / střední / rychlý)  
**Podmínky:** kombinace `season`, `distance_cat`, `experience_cat`

---

### Parametry úlohy

| Parametr | Hodnota |
|---|---|
| Procedura | CF-Miner |
| Target | speed_cat |
| Base (min. počet záznamů) | 1000 |
| RelMax_leq (max. relativní podíl dominantní kategorie) | 0.45 |
| Cond atributy | season, distance_cat, experience_cat (maxlen=3) |
| Data | ultra_clean_cm.parquet (~6.87M záznamů) |

> ⚠️ **Metodická poznámka:** CF-Miner hledá podmínky, za kterých je
> cílový atribut (speed_cat) rozložen odlišně od celkového průměru.
> `RelMax_leq` ≤ 0.45 zajišťuje, že žádná kategorie nedominuje
> absolutně — hledáme skutečně různorodé profily.

## 1. Import a načtení dat

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from cleverminer import cleverminer
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/processed')

df_cm = pd.read_parquet(DATA_DIR / 'ultra_clean_cm.parquet')
print(f"Načteno: {len(df_cm):,} řádků")
print(f"Sloupce: {df_cm.columns.tolist()}")

## 2. Příprava dat pro úlohu

In [ ]:
# Pro tuto úlohu potřebujeme: season, distance_cat, experience_cat, speed_cat
cols = ['season', 'distance_cat', 'experience_cat', 'speed_cat']
df_task = df_cm[cols].dropna().copy()

print(f"Záznamy s kompletními daty: {len(df_task):,}")
print()
print("Rozložení season:")
print(df_task['season'].value_counts())
print()
print("Rozložení distance_cat:")
print(df_task['distance_cat'].value_counts())
print()
print("Rozložení experience_cat:")
print(df_task['experience_cat'].value_counts())
print()
print("Rozložení speed_cat (ověření ~33/33/33):")
print((df_task['speed_cat'].value_counts() / len(df_task) * 100).round(1))

## 3. CleverMiner — CF-Miner úloha

In [ ]:
cm = cleverminer(df=df_task)

cm.mine(
    proc='CFMiner',
    target='speed_cat',
    quantifiers={'Base': 1000, 'RelMax_leq': 0.45},
    cond={
        'attributes': [
            {'name': 'season',         'type': 'subset', 'minlen': 1, 'maxlen': 1},
            {'name': 'distance_cat',   'type': 'subset', 'minlen': 1, 'maxlen': 1},
            {'name': 'experience_cat', 'type': 'subset', 'minlen': 1, 'maxlen': 1},
        ],
        'minlen': 1, 'maxlen': 3, 'type': 'con'
    }
)

print("\nSouhrn:")
cm.print_summary()

## 4. Výsledky

In [ ]:
print("Všechna pravidla (seřazená dle RelMax):")
cm.print_rulelist(sortby='relmax', storesorted=True)

## 5. Extrakce pravidel pro analýzu

In [ ]:
rules = []
n = cm.get_rulecount()

for i in range(1, n + 1):
    quant = cm.get_quantifiers(i)
    rule_text = cm.get_ruletext(i)

    # Parsování podmínek z textu pravidla
    season_match = re.search(r'season\((\w+)\)', rule_text)
    dist_match   = re.search(r'distance_cat\((\w+)\)', rule_text)
    exp_match    = re.search(r'experience_cat\((\w+)\)', rule_text)

    # Profil speed_cat — extrakce ze slovníku kvantifikátorů
    rules.append({
        'rule_id':    i,
        'season':     season_match.group(1) if season_match else None,
        'distance':   dist_match.group(1)   if dist_match   else None,
        'experience': exp_match.group(1)    if exp_match    else None,
        'base':       quant.get('base'),
        'relmax':     quant.get('relmax'),
        'rule_text':  rule_text,
    })

df_rules = pd.DataFrame(rules)
print(f"Extrahováno {len(df_rules)} pravidel")
print()
print(df_rules.sort_values('relmax').to_string(index=False))

## 6. Vizualizace

In [ ]:
season_order  = ['jaro', 'léto', 'podzim', 'zima']
season_colors = {'jaro': '#70ad47', 'léto': '#e07b54', 'podzim': '#f4b942', 'zima': '#5b9bd5'}
dist_order    = ['kratka', 'stredni', 'dlouha', 'extremni', 'casovy']
dist_labels   = {'kratka': '<60 km', 'stredni': '60-100 km',
                 'dlouha': '100-170 km', 'extremni': '>170 km', 'casovy': 'časový'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('CF-Miner: Sezónní profil výkonnosti\n'
             '(target: speed_cat, RelMax ≤ 0.45)',
             fontsize=13, fontweight='bold')

# Graf 1 — Počet pravidel dle sezóny
season_counts = df_rules['season'].value_counts().reindex(season_order, fill_value=0)
bars = ax1.bar(season_order,
               [season_counts.get(s, 0) for s in season_order],
               color=[season_colors[s] for s in season_order],
               alpha=0.85, edgecolor='white')
for bar in bars:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 0.1,
             str(int(h)), ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_xlabel('Sezóna', fontsize=11)
ax1.set_ylabel('Počet splněných pravidel', fontsize=11)
ax1.set_title('Počet CF-Miner pravidel dle sezóny\n'
              '(čím více pravidel, tím diverzifikovanější profil)', fontsize=11)
ax1.grid(axis='y', alpha=0.3)

# Graf 2 — RelMax dle vzdálenosti pro pravidla se sezónou (pokud existují)
df_with_dist = df_rules[df_rules['distance'].notna()].copy()
if len(df_with_dist) > 0:
    df_with_dist['distance_ord'] = pd.Categorical(
        df_with_dist['distance'], categories=dist_order, ordered=True
    )
    df_with_dist = df_with_dist.sort_values('distance_ord')

    # Průměrný RelMax dle vzdálenosti
    relmax_by_dist = df_with_dist.groupby('distance_ord')['relmax'].mean()
    dist_cats = [d for d in dist_order if d in relmax_by_dist.index]
    x = np.arange(len(dist_cats))

    bars2 = ax2.bar(x, [relmax_by_dist.get(d, 0) for d in dist_cats],
                    color='#5b9bd5', alpha=0.85, edgecolor='white')
    for bar in bars2:
        h = bar.get_height()
        if h > 0:
            ax2.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=9)

    ax2.axhline(0.333, color='black', linewidth=1, linestyle='--', label='Rovnoměrné rozložení (33%)')
    ax2.axhline(0.45,  color='red',   linewidth=1, linestyle=':', label='Práh RelMax_leq=0.45')
    ax2.set_xlabel('Kategorie vzdálenosti', fontsize=11)
    ax2.set_ylabel('Průměrné RelMax', fontsize=11)
    ax2.set_title('Průměrný RelMax dle vzdálenosti\n'
                  '(nižší = vyrovnanější profil speed_cat)', fontsize=11)
    ax2.set_xticks(x)
    ax2.set_xticklabels([dist_labels.get(d, d) for d in dist_cats], fontsize=9)
    ax2.legend(fontsize=9)
    ax2.grid(axis='y', alpha=0.3)
    ax2.set_ylim(0.25, 0.50)
else:
    ax2.text(0.5, 0.5, 'Žádná pravidla s distance_cat', ha='center', va='center',
             transform=ax2.transAxes, fontsize=12)
    ax2.set_title('RelMax dle vzdálenosti', fontsize=11)

plt.tight_layout()
plt.savefig(DATA_DIR / '05_sezonni_profil.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graf uložen.")

## 7. Interpretace grafů

**Graf vlevo — Počet pravidel dle sezóny:**

Čím více pravidel CF-Miner nalezl pro danou sezónu, tím více
kombinací podmínek vede k diverzifikovanému profilu speed_cat.
Sezóna s nejvíce pravidly má nejrůznorodější výkonnostní segmenty.

**Graf vpravo — Průměrný RelMax dle vzdálenosti:**

RelMax blízký 0.33 znamená rovnoměrné rozložení (žádná kategorie
speed_cat výrazně nedominuje). RelMax blízký 0.45 (práh) znamená,
že jedna kategorie tvoří 45 % závodníků — mírně nerovnoměrný profil.
Nižší RelMax = vyrovnanější soutěžní pole.

## 8. Zajímavá pravidla

In [ ]:
print("=== ZAJÍMAVÁ PRAVIDLA ===")
print()

# Top 3 pravidla s nejnižším RelMax (nejrovnoměrnější profil)
df_sorted = df_rules.sort_values('relmax')
print("TOP 3 pravidla (nejnižší RelMax — nejrovnoměrnější profil speed_cat):")
for _, row in df_sorted.head(3).iterrows():
    cm.print_rule(int(row['rule_id']))
    print()

## 9. Souhrn a business interpretace

In [ ]:
print("=" * 60)
print("SOUHRN — Úloha 1 (CF): Sezónní profil výkonnosti")
print("=" * 60)
print()

print(f"Celkem nalezených pravidel: {len(df_rules)}")
print()

# Sezónní rozdělení pravidel
print("Pravidla dle sezóny:")
for season in season_order:
    cnt = len(df_rules[df_rules['season'] == season])
    print(f"  {season:8s}: {cnt} pravidel")
print()

# Nejzajímavější kombinace
if len(df_rules) > 0:
    best = df_rules.sort_values('relmax').iloc[0]
    print(f"Nejrovnoměrnější profil: {best['rule_text']}")
    print(f"  RelMax = {best['relmax']:.3f}, Base = {best['base']:,.0f}")

print()
print("BUSINESS DOPORUČENÍ:")
print("  → Sezóny s diverzifikovanějším profilem vhodné pro začátečníky")
print("  → Zimní závody s homogenním profilem jsou doménou elitních závodníků")
print("  → Plánování kalendáře závodů dle cílové skupiny závodníků")

## Shrnutí

**Metoda:** CF-Miner (CleverMiner 1.2.6). Hledáme podmínky (kombinace
sezóny, vzdálenosti, zkušenosti) kde `speed_cat` má diverzifikované
rozložení (RelMax ≤ 0.45, Base ≥ 1000).

**Data:** ~6.87M závodníků, speed_cat per event.

**Klíčový nález:** Sezóna ovlivňuje výkonnostní profil závodního pole —
určité kombinace sezóna × vzdálenost × zkušenost vedou k výrazně
odlišnému složení závodníků dle tempa.

**Limitace:**
- Sezóna je odvozena z data závodu — neodráží aktuální počasí
- RelMax_leq = 0.45 je mírně nad 1/3 — pravidla mohou být jen slabě odlišná od průměru
- Chybějící záznamy pro surface a elevation snižují počet kombinací

**Další notebook:** `06_CF_uloha2.ipynb` — Profilování dle věku a povrchu (CF-Miner)